<a href="https://colab.research.google.com/github/wasuarezm-cell/optimizacion-ultima-milla---Simulated/blob/main/Analisis_Tiempos_de_Entrega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42) # Para reproducibilidad

# ==========================================
# 1. DIMENSIONES: Transportistas y Geografía
# ==========================================
dim_transportistas = pd.DataFrame([
    {'transportista_id': 'TR-01', 'nombre': 'Coordinadora', 'sla_prometido_dias': 3},
    {'transportista_id': 'TR-02', 'nombre': 'Servientrega', 'sla_prometido_dias': 2},
    {'transportista_id': 'TR-03', 'nombre': 'TCC', 'sla_prometido_dias': 4},
    {'transportista_id': 'TR-04', 'nombre': 'Envía', 'sla_prometido_dias': 3}
])

dim_geografia = pd.DataFrame({
    'ciudad_id': ['BOG', 'MED', 'CAL', 'BGA', 'BAQ'],
    'ciudad': ['Bogotá', 'Medellín', 'Cali', 'Bucaramanga', 'Barranquilla'],
    'region': ['Centro', 'Antioquia', 'Pacífico', 'Oriente', 'Caribe'],
    'distancia_km': [0, 415, 460, 395, 1000] # Asumiendo bodega central en Bogotá
})

In [ ]:
# ==========================================
# 2. HECHOS: Generación de Órdenes y Tiempos
# ==========================================
n_ordenes = 8500
fecha_inicio = datetime(2026, 1, 1)
fechas_base = [fecha_inicio + timedelta(days=np.random.randint(0, 180), hours=np.random.randint(8, 20)) for _ in range(n_ordenes)]

ordenes_data = []

for i in range(n_ordenes):
    orden_id = f'ORD-{10000 + i}'
    fecha_orden = fechas_base[i]
    ciudad_destino = np.random.choice(dim_geografia['ciudad_id'], p=[0.35, 0.25, 0.15, 0.15, 0.10])
    transp_id = np.random.choice(dim_transportistas['transportista_id'])

    # TIEMPO 1: Preparación en Almacén (Picking & Packing)
    # Regla Oculta 1: Fines de semana el almacén colapsa
    dias_almacen = 1
    if fecha_orden.weekday() in [4, 5]: # Viernes o Sábado
        dias_almacen = np.random.randint(3, 5)

    fecha_despacho = fecha_orden + timedelta(days=dias_almacen, hours=np.random.randint(2, 12))

    # TIEMPO 2: Tránsito del Transportista (Última Milla)
    # Regla base: depende de la distancia
    if ciudad_destino == 'BOG': dias_transito = 1
    elif ciudad_destino == 'BAQ': dias_transito = 3
    else: dias_transito = 2

    # Regla Oculta 2: Envía sufre en Santander (Bucaramanga)
    if transp_id == 'TR-04' and ciudad_destino == 'BGA':
        dias_transito += np.random.randint(3, 6)

    # Regla Oculta 3: Servientrega incumple rutas largas (Barranquilla)
    if transp_id == 'TR-02' and ciudad_destino == 'BAQ' and np.random.rand() > 0.6:
        dias_transito += np.random.randint(2, 4)

    # Añadir variabilidad normal
    dias_transito += np.random.choice([0, 0, 1, -1])
    if dias_transito < 1: dias_transito = 1 # Mínimo 1 día

    fecha_entrega = fecha_despacho + timedelta(days=int(dias_transito), hours=int(np.random.randint(-4, 6)))

    # Estado final
    estado = 'Entregado'
    if np.random.rand() < 0.02: # 2% de paquetes perdidos o devueltos
        estado = np.random.choice(['Devuelto', 'Perdido'])
        fecha_entrega = pd.NaT

    ordenes_data.append({
        'orden_id': orden_id,
        'fecha_orden': fecha_orden,
        'fecha_despacho': fecha_despacho,
        'fecha_entrega_real': fecha_entrega,
        'ciudad_id': ciudad_destino,
        'transportista_id': transp_id,
        'estado_envio': estado
    })

fact_envios = pd.DataFrame(ordenes_data)

In [ ]:
# ==========================================
# 3. EXPORTAR A CSV
# ==========================================
dim_transportistas.to_csv('dim_transportistas.csv', index=False)
dim_geografia.to_csv('dim_geografia.csv', index=False)
fact_envios.to_csv('fact_envios.csv', index=False)

print(f"Bases de datos generadas con éxito:")
print(f"- dim_transportistas: {len(dim_transportistas)} registros")
print(f"- dim_geografia: {len(dim_geografia)} registros")
print(f"- fact_envios: {len(fact_envios)} registros procesados")

Bases de datos generadas con éxito:
- dim_transportistas: 4 registros
- dim_geografia: 5 registros
- fact_envios: 8500 registros procesados


In [ ]:
import sqlite3

# 1. Crear la conexión a una base de datos local (se creará el archivo .db)
conn = sqlite3.connect('logistica_ecommerce.db')

# 2. Subir los datos a tablas SQL
dim_transportistas.to_sql('dim_transportistas', conn, if_exists='replace', index=False)
dim_geografia.to_sql('dim_geografia', conn, if_exists='replace', index=False)
fact_envios.to_sql('fact_envios', conn, if_exists='replace', index=False)

print("¡Base de datos SQL lista para consultar!")

¡Base de datos SQL lista para consultar!


In [ ]:
consulta_1 = """
/*
====================================================================
CONSULTA 1: Rendimiento General por Transportista (SLA vs Realidad)
Objetivo: Identificar si los proveedores logísticos están cumpliendo
          su Acuerdo de Nivel de Servicio (SLA) a nivel macro.
====================================================================
*/
SELECT
    t.nombre AS Empresa_Transporte,
    COUNT(f.orden_id) AS Total_Envios,
    t.sla_prometido_dias AS Promesa_SLA_Dias,
    ROUND(AVG(julianday(f.fecha_entrega_real) - julianday(f.fecha_despacho)), 1) AS Tiempo_Promedio_Real
FROM fact_envios f
JOIN dim_transportistas t ON f.transportista_id = t.transportista_id
WHERE f.estado_envio = 'Entregado'
GROUP BY t.nombre
ORDER BY Tiempo_Promedio_Real DESC;
"""

resultado_1 = pd.read_sql(consulta_1, conn)
display(resultado_1)

,Empresa_Transporte,Total_Envios,Promesa_SLA_Dias,Tiempo_Promedio_Real
0,Envía,2155,3,2.4
1,Servientrega,2079,2,2.0
2,TCC,2066,4,1.9
3,Coordinadora,2049,3,1.9


In [ ]:
consulta_2 = """
/*
====================================================================
CONSULTA 2: Desglose del Tiempo de Ciclo (Cycle Time)
Objetivo: Separar el tiempo operativo interno (nuestro almacén) del
          tiempo logístico externo (transportista) para aislar el cuello de botella.
====================================================================
*/
SELECT
    t.nombre AS Empresa_Transporte,
    ROUND(AVG(julianday(f.fecha_despacho) - julianday(f.fecha_orden)), 1) AS Tiempo_Promedio_Almacen,
    ROUND(AVG(julianday(f.fecha_entrega_real) - julianday(f.fecha_despacho)), 1) AS Tiempo_Promedio_Transito,
    ROUND(AVG(julianday(f.fecha_entrega_real) - julianday(f.fecha_orden)), 1) AS Tiempo_Ciclo_Total
FROM fact_envios f
JOIN dim_transportistas t ON f.transportista_id = t.transportista_id
WHERE f.estado_envio = 'Entregado'
GROUP BY t.nombre
ORDER BY Tiempo_Ciclo_Total DESC;
"""

resultado_2 = pd.read_sql(consulta_2, conn)
display(resultado_2)

,Empresa_Transporte,Tiempo_Promedio_Almacen,Tiempo_Promedio_Transito,Tiempo_Ciclo_Total
0,Envía,2.0,2.4,4.3
1,TCC,2.0,1.9,3.9
2,Servientrega,1.9,2.0,3.9
3,Coordinadora,2.0,1.9,3.9


In [ ]:
consulta_3 = """
/*
====================================================================
CONSULTA 3: Rendimiento Interno por Día de la Semana
Objetivo: Detectar si el almacén sufre colapsos operativos en
          días específicos (Análisis de estacionalidad semanal).
====================================================================
*/
SELECT
    CASE strftime('%w', fecha_orden)
        WHEN '0' THEN 'Domingo'
        WHEN '1' THEN 'Lunes'
        WHEN '2' THEN 'Martes'
        WHEN '3' THEN 'Miércoles'
        WHEN '4' THEN 'Jueves'
        WHEN '5' THEN 'Viernes'
        WHEN '6' THEN 'Sábado'
    END AS Dia_Compra,
    COUNT(orden_id) AS Volumen_Ordenes,
    ROUND(AVG(julianday(fecha_despacho) - julianday(fecha_orden)), 1) AS Dias_Promedio_Empaque
FROM fact_envios
GROUP BY Dia_Compra
ORDER BY Dias_Promedio_Empaque DESC;
"""

resultado_3 = pd.read_sql(consulta_3, conn)
display(resultado_3)

,Dia_Compra,Volumen_Ordenes,Dias_Promedio_Empaque
0,Viernes,1176,3.8
1,Sábado,1258,3.8
2,Miércoles,1190,1.3
3,Martes,1174,1.3
4,Lunes,1204,1.3
5,Jueves,1250,1.3
6,Domingo,1248,1.3


In [ ]:
consulta_4 = """
/*
====================================================================
CONSULTA 4: Mapa de Retrasos Críticos (Transportista + Ciudad)
Objetivo: Encontrar fallas de última milla cruzando el proveedor
          logístico con la ruta geográfica de destino.
====================================================================
*/
SELECT
    t.nombre AS Empresa_Transporte,
    g.ciudad AS Ciudad_Destino,
    COUNT(f.orden_id) AS Volumen_Envios,
    t.sla_prometido_dias AS Promesa_SLA_Dias,
    ROUND(AVG(julianday(f.fecha_entrega_real) - julianday(f.fecha_despacho)), 1) AS Tiempo_Transito_Real,
    ROUND(AVG(julianday(f.fecha_entrega_real) - julianday(f.fecha_despacho)) - t.sla_prometido_dias, 1) AS Dias_Retraso
FROM fact_envios f
JOIN dim_transportistas t ON f.transportista_id = t.transportista_id
JOIN dim_geografia g ON f.ciudad_id = g.ciudad_id
WHERE f.estado_envio = 'Entregado'
GROUP BY t.nombre, g.ciudad
ORDER BY Dias_Retraso DESC
LIMIT 10;
"""

resultado_4 = pd.read_sql(consulta_4, conn)
display(resultado_4)

,Empresa_Transporte,Ciudad_Destino,Volumen_Envios,Promesa_SLA_Dias,Tiempo_Transito_Real,Dias_Retraso
0,Envía,Bucaramanga,311,3,6.0,3.0
1,Servientrega,Barranquilla,208,2,4.2,2.2
2,Servientrega,Bucaramanga,307,2,2.1,0.1
3,Servientrega,Cali,323,2,2.1,0.1
4,Coordinadora,Barranquilla,236,3,3.0,0.0
5,Envía,Barranquilla,203,3,3.0,-0.0
6,Servientrega,Medellín,532,2,2.0,0.0
7,Servientrega,Bogotá,709,2,1.2,-0.8
8,Coordinadora,Bucaramanga,302,3,2.0,-1.0
9,Coordinadora,Cali,319,3,2.0,-1.0
